<a href="https://colab.research.google.com/github/labonig/CMSC-487-HW-4/blob/main/_downloads/4e865243430a47a00d551ca0579a6f6c/cifar10_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

References:
- [cifar10_tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- The homework instructions
- https://www.analyticsvidhya.com/blog/2021/08/all-you-need-to-know-about-skip-connections/

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
criterion = nn.CrossEntropyLoss()

### Function Definitions

In [3]:
# Define the data augmentation function
def get_data_augmentation_transform():
    import torchvision.transforms as transforms
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),  # Randomly crop the image to 32x32 with 4-pixel padding
        transforms.RandomRotation(0),          # Randomly rotate the image by up to 15 degrees
        transforms.ToTensor(),                  # Convert image to a PyTorch tensor
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    return transform

In [4]:
def train_network(nn_name, optimizer_name, num_epochs, is_augmented):
  correct = 0
  total = 0
  for epoch in range(num_epochs):  # loop over the dataset multiple times
      running_loss = 0.0
      for i, data in enumerate(trainloader, 0):
          # get the inputs; data is a list of [inputs, labels]
          inputs, labels = data

          if is_augmented == True:
            # Obtain the data augmentation transform
            transform = get_data_augmentation_transform()

            # Apply the transform to the image.
            pil_images = transforms.functional.to_pil_image(images[1])
            transformed_images = transform(pil_images)
            images[1] = transformed_images

          # zero the parameter gradients
          optimizer_name.zero_grad()

          # forward + backward + optimize
          outputs = nn_name(inputs)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer_name.step()

          # print statistics
          running_loss += loss.item()
          if i % 2000 == 1999:    # print every 2000 mini-batches
              print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
              running_loss = 0.0

          # calculate the training accuracy in the last epoch
          if epoch == num_epochs-1:
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

  training_score = 100 * correct // total
  print('Finished Training')
  print(f'Training accuracy: {training_score} %')


In [5]:
# function to calculate the percentage of correct predictions on the test data
def test_network(nn_name):
  correct = 0
  total = 0
  # since we're not training, we don't need to calculate the gradients for our outputs
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          # calculate outputs by running images through the network
          outputs = nn_name(images)
          # the class with the highest energy is what we choose as prediction
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  test_score = 100 * correct // total
  print(f'Accuracy of the network on the 10000 test images: {test_score} %')

In [6]:
def print_accuracy(nn_name):
  # prepare to count predictions for each class
  correct_pred = {classname: 0 for classname in classes}
  total_pred = {classname: 0 for classname in classes}

  # again no gradients needed
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          outputs = nn_name(images)
          _, predictions = torch.max(outputs, 1)
          # collect the correct predictions for each class
          for label, prediction in zip(labels, predictions):
              if label == prediction:
                  correct_pred[classes[label]] += 1
              total_pred[classes[label]] += 1

  # print accuracy for each class
  for classname, correct_count in correct_pred.items():
      accuracy = 100 * float(correct_count) / total_pred[classname]
      print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

### Load and normalize CIFAR10


In [7]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:03<00:00, 49.4MB/s]


In [8]:
dataiter = iter(testloader)
images, labels = next(dataiter)

Q0 - Training a Classifier (Basic CNN)
========================================


In [9]:
# Question 0: Basic CNN
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

In [10]:
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [11]:
# Training the basic network
train_network(net, optimizer, 10, False)

[1,  2000] loss: 2.196
[1,  4000] loss: 1.841
[1,  6000] loss: 1.686
[1,  8000] loss: 1.595
[1, 10000] loss: 1.552
[1, 12000] loss: 1.484
[2,  2000] loss: 1.428
[2,  4000] loss: 1.421
[2,  6000] loss: 1.364
[2,  8000] loss: 1.361
[2, 10000] loss: 1.332
[2, 12000] loss: 1.299
[3,  2000] loss: 1.261
[3,  4000] loss: 1.257
[3,  6000] loss: 1.220
[3,  8000] loss: 1.226
[3, 10000] loss: 1.209
[3, 12000] loss: 1.191
[4,  2000] loss: 1.122
[4,  4000] loss: 1.136
[4,  6000] loss: 1.146
[4,  8000] loss: 1.128
[4, 10000] loss: 1.104
[4, 12000] loss: 1.157
[5,  2000] loss: 1.056
[5,  4000] loss: 1.064
[5,  6000] loss: 1.068
[5,  8000] loss: 1.056
[5, 10000] loss: 1.077
[5, 12000] loss: 1.073
[6,  2000] loss: 1.003
[6,  4000] loss: 0.970
[6,  6000] loss: 0.993
[6,  8000] loss: 1.021
[6, 10000] loss: 1.040
[6, 12000] loss: 1.014
[7,  2000] loss: 0.922
[7,  4000] loss: 0.954
[7,  6000] loss: 0.943
[7,  8000] loss: 0.965
[7, 10000] loss: 0.994
[7, 12000] loss: 0.984
[8,  2000] loss: 0.873
[8,  4000] 

In [12]:
# save the trained model:
PATH = "cifar_net.pt"
torch.save(net.state_dict(), PATH)

In [13]:
# Testing accuracy of the basic network
test_network(net)

Accuracy of the network on the 10000 test images: 60 %


In [14]:
# print the per-class accuracy of the tutorial network
print_accuracy(net)

Accuracy for class: plane is 70.3 %
Accuracy for class: car   is 61.6 %
Accuracy for class: bird  is 51.3 %
Accuracy for class: cat   is 50.7 %
Accuracy for class: deer  is 48.0 %
Accuracy for class: dog   is 47.7 %
Accuracy for class: frog  is 59.2 %
Accuracy for class: horse is 64.5 %
Accuracy for class: ship  is 76.3 %
Accuracy for class: truck is 76.9 %


Q1 - Deeper CNN Architecture
========================================


In [31]:
# Question 1: Deeper CNN architecture
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5, padding='same')
        self.conv2 = nn.Conv2d(6, 16, 7, padding='same')
        self.conv3 = nn.Conv2d(16, 32, 5, padding='same')
        self.conv4 = nn.Conv2d(32, 64, 7, padding='same')
        self.conv5 = nn.Conv2d(64, 128, 9, padding='same')
        self.conv6 = nn.Conv2d(128, 150, 11, padding='same')
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(150 * 8 * 8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # shape: 3 x 32 x 32
        # convolutional block 1
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = self.pool(x)
        # shape: 16 x 16 x 16

        # convolutional block 2
        x = F.softmax(self.conv3(x))
        x = F.softmax(self.conv4(x))
        x = F.softmax(self.conv5(x))
        x = F.softmax(self.conv6(x))

        x = self.pool(x)
        # shape: 64 x 8 x 8

        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

deeper_net = Net()

In [32]:
deep_optimizer = optim.SGD(deeper_net.parameters(), lr=0.001, momentum=0.9)

In [33]:
# Training the deeper model with convolutional blocks
train_network(deeper_net, deep_optimizer, 10, False)


[1,  2000] loss: 2.288
[1,  4000] loss: 1.899
[1,  6000] loss: 1.661
[1,  8000] loss: 1.601
[1, 10000] loss: 1.535
[1, 12000] loss: 1.475
[2,  2000] loss: 1.417
[2,  4000] loss: 1.378
[2,  6000] loss: 1.345
[2,  8000] loss: 1.336
[2, 10000] loss: 1.302
[2, 12000] loss: 1.294
[3,  2000] loss: 1.232
[3,  4000] loss: 1.227
[3,  6000] loss: 1.200
[3,  8000] loss: 1.193
[3, 10000] loss: 1.157
[3, 12000] loss: 1.178
[4,  2000] loss: 1.109
[4,  4000] loss: 1.115
[4,  6000] loss: 1.103
[4,  8000] loss: 1.098
[4, 10000] loss: 1.093
[4, 12000] loss: 1.101
[5,  2000] loss: 1.008
[5,  4000] loss: 1.029
[5,  6000] loss: 1.012
[5,  8000] loss: 1.018
[5, 10000] loss: 1.044
[5, 12000] loss: 1.032
[6,  2000] loss: 0.946
[6,  4000] loss: 0.974
[6,  6000] loss: 0.964
[6,  8000] loss: 0.973
[6, 10000] loss: 0.977
[6, 12000] loss: 0.985
[7,  2000] loss: 0.905
[7,  4000] loss: 0.912
[7,  6000] loss: 0.918
[7,  8000] loss: 0.924
[7, 10000] loss: 0.948
[7, 12000] loss: 0.937
[8,  2000] loss: 0.831
[8,  4000] 

In [34]:
# save the trained model:
PATH2 = "cifar_deeper_net.pt"
torch.save(deeper_net.state_dict(), PATH2)

In [35]:
# Testing accuracy of the deeper network
test_network(deeper_net)

Accuracy of the network on the 10000 test images: 62 %


In [36]:
# print the per-class accuracy of the deeper network
print_accuracy(deeper_net)

Accuracy for class: plane is 71.6 %
Accuracy for class: car   is 73.8 %
Accuracy for class: bird  is 44.9 %
Accuracy for class: cat   is 37.8 %
Accuracy for class: deer  is 54.4 %
Accuracy for class: dog   is 45.9 %
Accuracy for class: frog  is 74.9 %
Accuracy for class: horse is 70.3 %
Accuracy for class: ship  is 77.0 %
Accuracy for class: truck is 77.1 %


Q2 - Implementing Residual Connections
========================================

In [21]:
# copied from the HW instructions
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel, stride=1):
        super(ResidualBlock, self).__init__()
        # First convolutional layer
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=kernel,
                               stride=stride, padding='same', bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        # Second convolutional layer
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=kernel,
                               stride=1, padding='same', bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Downsample layer for matching dimensions, if needed
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        # Add the residual (skip) connection
        out += identity
        out = self.relu(out)

        return out

In [22]:
# define and train the same CNN as Q1 but with residual connections at the end of each convolutional block
resnet_blocks = nn.Sequential(
    nn.Conv2d(3, 6, kernel_size=5, stride=1, padding='same', bias=False),
    nn.ReLU(inplace=True),
    ResidualBlock(in_channels=6, out_channels=16, kernel=7, stride=1),

    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(16, 32, kernel_size=5, stride=1, padding='same', bias=False),
    nn.ReLU(inplace=True),
    nn.Conv2d(32, 64, kernel_size=7, stride=1, padding='same', bias=False),
    nn.ReLU(inplace=True),
    nn.Conv2d(64, 128, kernel_size=9, stride=1, padding='same', bias=False),
    nn.ReLU(inplace=True),
    ResidualBlock(in_channels=128, out_channels=150, kernel=11, stride=1),

    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),

    nn.Linear(150 * 8 * 8, 120),
    nn.ReLU(inplace=True),
    nn.Linear(120, 84),
    nn.ReLU(inplace=True),
    nn.Linear(84, 10)
)

In [23]:
res_optimizer = optim.SGD(resnet_blocks.parameters(), lr=0.001, momentum=0.9)

In [24]:
# Training the residual connections model
train_network(resnet_blocks, res_optimizer, 10, False)

[1,  2000] loss: 2.046
[1,  4000] loss: 1.806


KeyboardInterrupt: 

In [25]:
# save the trained model:
PATH3 = "cifar_res_block.pt"
torch.save(resnet_blocks.state_dict(), PATH3)

In [27]:
# Testing accuracy of the residual connections cnn
test_network(resnet_blocks)

KeyboardInterrupt: 

In [ ]:
# print the per-class accuracy of the residual/skip connection network
print_accuracy(resnet_blocks)

Q3 - Data Augmentation
========================================


In [28]:
aug_deeper_net = DeepNet()

In [29]:
aug_optimizer = optim.SGD(aug_deeper_net.parameters(), lr=0.001, momentum=0.9)

In [30]:
train_network(aug_deeper_net, aug_optimizer, 10, True)

[1,  2000] loss: 2.303


KeyboardInterrupt: 

In [37]:
# save the trained model:
PATH4 = "cifar_aug_deeper_net.pt"
torch.save(aug_deeper_net.state_dict(), PATH4)

In [ ]:
# Testing accuracy of the augmented-data, deeper cnn
test_network(aug_deeper_net)

In [ ]:
# print the per-class accuracy of the residual/skip connection network
print_accuracy(aug_deeper_net)

In [ ]:
# Clean up
del dataiter